# التمثيل العميق للسمات باستخدام EfficientNet-B6 — D-MorphNet

هذا الدفتر ينفّذ خطوة **"Deep Feature Representation Using EfficientNet-B6"** — في الإطار المحدَّث استُبدل EfficientNet-B5 بـ **EfficientNet-B6** كعمود فقري لاستخراج السمات، بأسلوب **التعلم بالنقل (Transfer Learning)**.

## الأساس الرياضي

**١) استخراج السمات:** إذا كانت صورة الوجه المدخلة $I \in \mathbb{R}^{H \times W \times C}$، فإن السمات العميقة تُستخرج بالدالة:

$$F = f_{B6}(I;\ \theta)$$

حيث $f_{B6}(\cdot)$ هي دالة EfficientNet-B6، و $\theta$ الأوزان المدرّبة مسبقاً على ImageNet، و $F \in \mathbb{R}^{d}$ متجه السمات الناتج (عندنا $d = 2304$).

**٢) داخل كل كتلة التفافية (Convolutional Block):**

$$X_l = \sigma(W_l * X_{l-1} + b_l)$$

حيث $X_{l-1}$ و $X_l$ خرائط السمات للدخل والخرج، $W_l$ و $b_l$ أوزان وانحياز الالتفاف، $*$ عملية الالتفاف، و $\sigma(\cdot)$ دالة التفعيل غير الخطية **Swish** المستخدمة في EfficientNet.

**٣) التجميع المتوسط الشامل (GAP)** يحوّل خرائط السمات المكانية إلى متجه ثابت الطول:

$$F = \frac{1}{N}\sum_{i=1}^{N} X_L^{(i)}$$

حيث $X_L$ خرائط السمات في آخر مرحلة التفافية و $N$ عدد المواقع المكانية.

## خطة التنفيذ (كما في الورقة)
1. تحميل B6 بأوزان ImageNet مع **حذف طبقات التصنيف النهائية** ← يعمل كمستخرج سمات فقط.
2. إضافة طبقة **GAP** لتحويل الخرج إلى متجه سمات يلتقط بنية الوجه ونسيجه وآثار المورف.
3. **المرحلة الأولى**: استخراج السمات والطبقات **مجمّدة بالكامل** (دون أي تعديل).
4. **المرحلة الثانية — الضبط الدقيق (Fine-tuning)**: نجعل **الطبقات العليا فقط** قابلة للتدريب مع تثبيت الطبقات المبكرة، فيتخصص النموذج في أنماط المورف **دون فقدان** السمات الوجهية العامة المتعلَّمة مسبقاً.
5. إرسال السمات النهائية إلى **مصنّف SVM** لاتخاذ القرار: حقيقية أم مدموجة.

> ⚙️ فعّل GPU قبل التشغيل. الضبط الدقيق لـ B6 بحجم 528×528 مكلف — الإعدادات أدناه مضبوطة لتناسب ذاكرة T4.


## ١) تجهيز البيئة

نفعّل أيضاً **الدقة المختلطة (Mixed Precision)** — تخفّض استهلاك ذاكرة GPU وتسرّع التدريب على T4 دون التأثير على الجودة.


In [ ]:
!pip -q install scikit-learn joblib tqdm

import os, glob, random, shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

gpus = tf.config.list_physical_devices('GPU')
print("TensorFlow:", tf.__version__)
print("GPU:", gpus if gpus else "⚠️ لا يوجد GPU — فعّله من إعدادات الجلسة!")

# الدقة المختلطة لتوفير الذاكرة وتسريع الضبط الدقيق
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("سياسة الدقة:", tf.keras.mixed_precision.global_policy().name)


## ٢) تحميل الصور المعالجة (528×528 + CLAHE)

نستخدم مخرجات دفتر المعالجة المسبقة — من الجلسة الحالية أو بفك الضغط من Google Drive.


In [ ]:
PROC_DIR = "/content/DMorphNet_processed"
PROC_ZIP = "/content/drive/MyDrive/DMorphNet_processed.zip"

if not os.path.isdir(os.path.join(PROC_DIR, "train")):
    from google.colab import drive
    drive.mount('/content/drive')
    print("جاري فك الضغط من Google Drive ...")
    shutil.unpack_archive(PROC_ZIP, PROC_DIR)

SPLITS = ["train", "val", "test"]
LABEL_MAP = {"real": 0, "morph": 1}      # 0 = حقيقية ، 1 = مدموجة

for split in SPLITS:
    counts = {c: len(glob.glob(os.path.join(PROC_DIR, split, c, "*.jpg")))
              for c in LABEL_MAP}
    print(f"{split}: حقيقية={counts['real']}  مدموجة={counts['morph']}")


## ٣) الإعدادات وخطوط قراءة البيانات

- خط **الاستخراج/التقييم**: قراءة ← `preprocess_input` الخاص بـ EfficientNet فقط.
- خط **الضبط الدقيق**: نفسه + زيادة بيانات خفيفة (قلب، سطوع/تباين، جودة JPEG عشوائية) لتقوية التعميم.


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 528
EXTRACT_BATCH = 16        # دفعة استخراج السمات
FT_BATCH = 4              # دفعة الضبط الدقيق (B6 بحجم 528 كبير على الذاكرة)
AUTOTUNE = tf.data.AUTOTUNE

FEAT_DIR = "/content/DMorphNet_features"
os.makedirs(FEAT_DIR, exist_ok=True)

from tensorflow.keras.applications.efficientnet import preprocess_input


def list_files(split):
    paths, labels = [], []
    for cls, lab in LABEL_MAP.items():
        fs = sorted(glob.glob(os.path.join(PROC_DIR, split, cls, "*.jpg")))
        paths += fs
        labels += [lab] * len(fs)
    return paths, np.array(labels, dtype=np.int32)


def decode(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.cast(img, tf.float32)
    return tf.ensure_shape(img, [IMG_SIZE, IMG_SIZE, 3])


def extract_ds(paths):
    # خط الاستخراج: بدون أي زيادة بيانات
    ds = tf.data.Dataset.from_tensor_slices(list(paths))
    ds = ds.map(lambda p: preprocess_input(decode(p)), num_parallel_calls=AUTOTUNE)
    return ds.batch(EXTRACT_BATCH).prefetch(AUTOTUNE)


def augment_tf(img):
    # زيادة بيانات خفيفة أثناء الضبط الدقيق فقط
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 25.0)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.clip_by_value(img, 0.0, 255.0)
    img = tf.image.random_jpeg_quality(tf.cast(img, tf.uint8), 40, 95)
    return tf.cast(img, tf.float32)


def finetune_ds(split, training=False):
    paths, labels = list_files(split)
    ds = tf.data.Dataset.from_tensor_slices((list(paths), labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED)

    def load(p, y):
        img = decode(p)
        if training:
            img = augment_tf(img)
        return preprocess_input(img), y

    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    return ds.batch(FT_BATCH).prefetch(AUTOTUNE)

print("✅ خطوط البيانات جاهزة")


## ٤) بناء النموذج وفق المعادلات

نبني المعمارية بحيث تطابق الصياغة الرياضية تماماً:
- `EfficientNetB6(include_top=False)` = حذف طبقة التصنيف النهائية ← الشبكة تعمل كدالة $f_{B6}(I;\theta)$ فقط (كل كتلة داخلية تنفّذ $X_l = \sigma(W_l * X_{l-1} + b_l)$ بتفعيل Swish).
- طبقة `GlobalAveragePooling2D` = معادلة GAP التي تحوّل خرائط $X_L$ إلى المتجه $F \in \mathbb{R}^{2304}$.
- رأس تصنيف مؤقت (Dropout + Dense/Sigmoid) نستخدمه **فقط أثناء الضبط الدقيق** ثم نتجاهله — القرار النهائي دائماً لـ SVM.


In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB6

# f_B6: العمود الفقري بأوزان ImageNet وبدون طبقات التصنيف
base = EfficientNetB6(include_top=False, weights="imagenet",
                      input_shape=(IMG_SIZE, IMG_SIZE, 3))

inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)                       # خرائط السمات X_L
gap = layers.GlobalAveragePooling2D(name="gap")(x)     # F = (1/N) Σ X_L
drop = layers.Dropout(0.3)(gap)
outputs = layers.Dense(1, activation="sigmoid", dtype="float32",
                       name="temp_head")(drop)         # رأس مؤقت للضبط الدقيق

clf_model  = Model(inputs, outputs, name="dmorphnet_finetune")
feat_model = Model(inputs, gap,     name="dmorphnet_features")

FEATURE_DIM = feat_model.output_shape[-1]
print(f"عدد طبقات B6: {len(base.layers)}")
print(f"طول متجه السمات F: {FEATURE_DIM}")


## ٥) المرحلة الأولى: الاستخراج بطبقات مجمّدة بالكامل

كما في الورقة: *"أولاً يُستخدم النموذج بطبقات ثابتة لاستخراج السمات دون أي تعديل"*.

نجمّد العمود الفقري كاملاً، نستخرج سمات الأقسام الثلاثة، ونحفظها في `.npz` (يُتخطى الاستخراج تلقائياً إن وُجدت ملفات محفوظة من دفتر سابق).


In [ ]:
base.trainable = False          # تجميد كامل — استخراج فقط

frozen_feats, labels_all, paths_all = {}, {}, {}
for split in SPLITS:
    paths, y = list_files(split)
    paths_all[split], labels_all[split] = paths, y
    out_path = os.path.join(FEAT_DIR, f"effb6_{split}.npz")
    if os.path.exists(out_path):
        frozen_feats[split] = np.load(out_path)["X"]
        print(f"{split}: سمات مجمّدة محمّلة من الملف {frozen_feats[split].shape}")
        continue
    print(f"استخراج سمات {split} (مجمّد) — {len(paths)} صورة ...")
    X = feat_model.predict(extract_ds(paths), verbose=1)
    frozen_feats[split] = X
    np.savez_compressed(out_path, X=X, y=y)

print("✅ اكتملت المرحلة الأولى (سمات الطبقات المجمّدة)")


## ٦) المرحلة الثانية: الضبط الدقيق للطبقات العليا فقط

نجعل **الكتلة الأخيرة (block7) والطبقات العلوية (top)** قابلة للتدريب ونُبقي كل الطبقات المبكرة مجمّدة:
- الطبقات المبكرة تحتفظ بالسمات الوجهية العامة (حواف، نسيج، بنية) المتعلَّمة من ImageNet.
- الطبقات العليا تتخصص في **أنماط وآثار المورف** الدقيقة.
- نُبقي طبقات **Batch Normalization مجمّدة** دائماً — تحريكها بدفعات صغيرة يفسد الإحصاءات المتعلَّمة.
- معدل تعلم **منخفض جداً** (1e-5) حتى لا ندمّر الأوزان المسبقة.


In [ ]:
FT_EPOCHS = 3
STEPS_PER_EPOCH = 1500      # عدد خطوات لكل حقبة (قلّله للتجربة السريعة)
VAL_STEPS = 300

# فتح الطبقات العليا فقط (block7 + top) مع إبقاء BatchNorm مجمّدة
base.trainable = True
n_trainable = 0
for layer in base.layers:
    if layer.name.startswith(("block7", "top")) and \
       not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True
        n_trainable += 1
    else:
        layer.trainable = False

print(f"الطبقات القابلة للتدريب: {n_trainable} من أصل {len(base.layers)}")

clf_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"])

history = clf_model.fit(
    finetune_ds("train", training=True),
    epochs=FT_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=finetune_ds("val"),
    validation_steps=VAL_STEPS)

# منحنيات التدريب
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="تدريب")
axes[0].plot(history.history["val_loss"], label="تحقق")
axes[0].set_title("الخسارة (Loss)"); axes[0].legend()
axes[1].plot(history.history["accuracy"], label="تدريب")
axes[1].plot(history.history["val_accuracy"], label="تحقق")
axes[1].set_title("الدقة (Accuracy)"); axes[1].legend()
plt.tight_layout()
plt.show()


## ٧) استخراج السمات بعد الضبط الدقيق

نتجاهل رأس التصنيف المؤقت ونستخدم `feat_model` (حتى طبقة GAP) لاستخراج متجهات $F$ الجديدة — هذه السمات الآن أكثر حساسية لآثار المورف مع احتفاظها بالمعرفة الوجهية العامة.


In [ ]:
ft_feats = {}
for split in SPLITS:
    out_path = os.path.join(FEAT_DIR, f"effb6_ft_{split}.npz")
    if os.path.exists(out_path):
        ft_feats[split] = np.load(out_path)["X"]
        print(f"{split}: سمات مضبوطة محمّلة {ft_feats[split].shape}")
        continue
    print(f"استخراج سمات {split} (بعد الضبط الدقيق) ...")
    X = feat_model.predict(extract_ds(paths_all[split]), verbose=1)
    ft_feats[split] = X
    np.savez_compressed(out_path, X=X, y=labels_all[split])

print("✅ سمات ما بعد الضبط الدقيق جاهزة")


## ٨) التصنيف النهائي بـ SVM ومقارنة المرحلتين

ندرّب SVM مرتين — على السمات **المجمّدة** وعلى سمات **ما بعد الضبط الدقيق** — ونقارن الدقة على التحقق والاختبار لإثبات فائدة الضبط الدقيق.

> نستخدم `LinearSVC` هنا للسرعة (2304 سمة تكفي لفصل خطي جيد). يمكنك التبديل إلى `SVC(kernel="rbf")` للنتيجة النهائية كما في دفتر تصميم النظام.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report


def train_eval_svm(feats, name):
    sc = StandardScaler().fit(feats["train"])
    clf = LinearSVC(C=1.0, random_state=SEED)
    clf.fit(sc.transform(feats["train"]), labels_all["train"])
    accs = {}
    for split in ["val", "test"]:
        pred = clf.predict(sc.transform(feats[split]))
        accs[split] = accuracy_score(labels_all[split], pred)
    print(f"{name}: دقة التحقق={accs['val']:.4f}  دقة الاختبار={accs['test']:.4f}")
    return clf, sc, accs


print("═══ المقارنة: مجمّد مقابل مضبوط دقيقاً ═══")
_, _, frozen_accs = train_eval_svm(frozen_feats, "B6 مجمّد بالكامل  ")
svm_ft, scaler_ft, ft_accs = train_eval_svm(ft_feats, "B6 بعد الضبط الدقيق")

gain = (ft_accs["test"] - frozen_accs["test"]) * 100
print(f"\nالتحسن في دقة الاختبار بعد الضبط الدقيق: {gain:+.2f} نقطة مئوية")

# تقرير تفصيلي للنظام النهائي (سمات مضبوطة + SVM)
pred_test = svm_ft.predict(scaler_ft.transform(ft_feats["test"]))
print("\nالتقرير النهائي على مجموعة الاختبار:")
print(classification_report(labels_all["test"], pred_test,
                            target_names=["حقيقية (real)", "مدموجة (morph)"]))


## ٩) حفظ النظام النهائي في Google Drive

نحفظ:
- العمود الفقري **المضبوط دقيقاً** (حتى طبقة GAP) بصيغة `.keras`.
- مصنّف SVM النهائي + الـ Scaler بصيغة `joblib`.
- ملفات السمات `.npz` للمرحلتين.

بهذا يكتمل مسار D-MorphNet: صورة ← $f_{B6}$ المضبوط ← GAP ← $F$ ← SVM ← **حقيقية / مدموجة**.


In [ ]:
import joblib
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/DMorphNet_models"
os.makedirs(SAVE_DIR, exist_ok=True)

feat_model.save(os.path.join(SAVE_DIR, "effb6_finetuned_features.keras"))
joblib.dump(svm_ft, os.path.join(SAVE_DIR, "svm_finetuned.joblib"))
joblib.dump(scaler_ft, os.path.join(SAVE_DIR, "scaler_finetuned.joblib"))
for split in SPLITS:
    for prefix in ["effb6", "effb6_ft"]:
        src = os.path.join(FEAT_DIR, f"{prefix}_{split}.npz")
        if os.path.exists(src):
            shutil.copy(src, SAVE_DIR)

print("تم الحفظ في:", SAVE_DIR)
print(os.listdir(SAVE_DIR))
print("\n🎉 اكتمل التمثيل العميق للسمات: B6 مضبوط دقيقاً + GAP + SVM")
